# INF2006 G12 - Task 4: Data analytics (volume anomalies, news signal)

**Inputs (Task 2 outputs, HDFS):** `/corpus/cleaned/prices/` (605,887 daily bars) and
`/corpus/cleaned/gdelt/daily_features/` (150 daily GDELT aggregates keyed by `available_date`, i.e. DATEADDED).
Q3 also reads the Task 3 2017 test predictions (volatility and direction models) and the Task 2 `model_ready_v1` splits.

**Questions**
- **Q1.** When is trading activity for a security unusually high or low versus its *own* history, and how does that relate to price movement?
- **Q2.** Do GDELT news-volume / tone groups differ in subsequent market volatility, and does news add anything beyond today's volatility?
- **Q3.** Where is the Task 3 model most and least reliable (by month and by volatility quartile) compared with the Task 3 baseline?

**Tools.** All heavy work (window functions, joins, group aggregates) runs in PySpark on the full tables. Only small aggregated
result tables (<= ~1,300 rows) are converted to pandas for charts and a Spearman check.

**Run order.** Run every cell top to bottom. Outputs go to `OUT_DIR` (default `evidence/task4/`) with fixed file names.

**Kernel note.** The notebook expects `spark` to exist in the same Python process that draws charts. If your kernel runs Spark remotely
(sparkmagic/Livy), plots may need `%matplot plt` or a local kernel; see the README note in the report appendix.

## EMR setup (PySpark kernel on JupyterHub) - run the next two cells first, once per fresh session

The cluster's default Python has no numpy/scipy/matplotlib for the Spark session. Cell 1 enables per-session package installs
(`%%configure -f` restarts the Spark session, so it must be the first cell and runs alone). Cell 2 installs the packages.
Skip both cells when running locally (they need the sparkmagic PySpark kernel).

In [1]:
%%configure -f
{"conf": {"spark.pyspark.python": "python3",
          "spark.pyspark.virtualenv.enabled": "true",
          "spark.pyspark.virtualenv.type": "native",
          "spark.pyspark.virtualenv.bin.path": "/usr/bin/virtualenv"}}

In [2]:
for pkg in ["numpy", "pandas", "scipy", "matplotlib"]:
    sc.install_pypi_package(pkg)

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1795160715899_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.8.1
    Not uninstalling python-dateutil at /usr/lib/python3.9/site-packages, outside environment /mnt/yarn/usercache/livy/appcache/application_1795160715899_0001/container_1795160715899_0001_01_000001/tmp/spark-3ff7f7f0-51fe-4544-b771-aac50b3353da
    Can't uninstall 'python-dateutil'. No files were found to uninstall.




ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
matplotlib 3.9.4 requires contourpy>=1.0.1, which is not installed.
matplotlib 3.9.4 requires cycler>=0.10, which is not installed.
matplotlib 3.9.4 requires fonttools>=4.22.0, which is not installed.
matplotlib 3.9.4 requires importlib-resources>=3.2.0; python_version < "3.10", which is not installed.
matplotlib 3.9.4 requires kiwisolver>=1.3.1, which is not installed.
matplotlib 3.9.4 requires pillow>=

In [3]:
import os, json, time, datetime
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F, Window

RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
T0 = time.time()
TIMINGS = {}

try:
    spark                      # provided by the EMR kernel
except NameError:              # local test only
    spark = (SparkSession.builder.master("local[2]").appName("g12_task4")
             .config("spark.ui.enabled", "false").getOrCreate())

ON_CLUSTER = not spark.sparkContext.master.startswith("local")
spark.conf.set("spark.sql.shuffle.partitions", "32" if ON_CLUSTER else "8")

# ---- parameters (change here only) ----
BUCKET      = os.environ.get("G12_BUCKET", "s3://<YOUR-BUCKET>")   # <- set to the S3 bucket that holds your copy of the inputs (no trailing slash)
assert "<YOUR-BUCKET>" not in BUCKET, "Set BUCKET (or env G12_BUCKET) to your S3 bucket before running"
BUCKET      = BUCKET.rstrip("/")
PRICES_PATH = os.environ.get("G12_PRICES", BUCKET + "/task2/cleaned/prices")
GDELT_PATH  = os.environ.get("G12_GDELT",  BUCKET + "/task2/cleaned/gdelt/daily_features")
OUT_DIR     = os.environ.get("G12_OUT_DIR", "evidence/task4")
WINDOW      = 20      # trading days of own history for the volume baseline
Z_HIGH, Z_LOW = 2.0, -2.0
MAX_ABS_RET = 0.5     # |daily return| above this is treated as a probable split/data artefact (set to NULL)
N_GROUPS    = 3       # terciles for news groups
os.makedirs(OUT_DIR, exist_ok=True)

def stage(name, t_start):
    TIMINGS[name] = round(time.time() - t_start, 1)
    print(f"[{name}] {TIMINGS[name]} s")

print("RUN_ID", RUN_ID, "| Spark", spark.version, "| on cluster:", ON_CLUSTER)
print("BUCKET     ", BUCKET); print("PRICES_PATH", PRICES_PATH); print("GDELT_PATH ", GDELT_PATH); print("OUT_DIR    ", OUT_DIR)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

RUN_ID 20261120T075457Z | Spark 3.5.5-amzn-1 | on cluster: True
BUCKET      s3://inf2006-g12-2500837
PRICES_PATH s3://inf2006-g12-2500837/task2/cleaned/prices
GDELT_PATH  s3://inf2006-g12-2500837/task2/cleaned/gdelt/daily_features
OUT_DIR     evidence/task4

## 0. Load inputs and check scope

In [4]:
t = time.time()
prices = spark.read.parquet(PRICES_PATH).select(
    "ticker", "trade_date", "open_price", "high_price", "low_price", "close_price", "volume")
gdelt = spark.read.parquet(GDELT_PATH)

n_prices, n_gdelt = prices.count(), gdelt.count()
scope = prices.agg(F.min("trade_date").alias("d0"), F.max("trade_date").alias("d1"),
                   F.countDistinct("ticker").alias("tickers"),
                   F.countDistinct("trade_date").alias("trading_days")).first()
g_scope = gdelt.agg(F.min("available_date").alias("d0"), F.max("available_date").alias("d1")).first()

EXPECT = {"price_rows": 605887, "gdelt_rows": 150}
print("price rows  :", n_prices, "PASS" if n_prices == EXPECT["price_rows"] else "FAIL (expected 605,887)")
print("gdelt rows  :", n_gdelt,  "PASS" if n_gdelt  == EXPECT["gdelt_rows"]  else "FAIL (expected 150)")
print("prices span :", scope["d0"], "to", scope["d1"], "|", scope["tickers"], "tickers |", scope["trading_days"], "trading days")
print("gdelt span  :", g_scope["d0"], "to", g_scope["d1"], "(three blocks: Jul-Sep 2015, Jan 2016, Jan 2017)")
stage("load_and_check", t)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

price rows  : 605887 PASS
gdelt rows  : 150 PASS
prices span : 2013-02-08 to 2017-12-29 | 505 tickers | 1233 trading days
gdelt span  : 2015-07-01 to 2017-01-30 (three blocks: Jul-Sep 2015, Jan 2016, Jan 2017)
[load_and_check] 10.6 s

## Q1. Unusual trading activity versus a security's own history

Method: for each ticker, `z = (log1p(volume) - mean) / std`, where mean and std use only the **previous 20 trading rows**
(today excluded, so a spike cannot inflate its own baseline). Log volume is used because raw volume is strongly right-skewed.
Days are labelled **high** (z >= 2), **low** (z <= -2) or **normal**. Daily returns with |return| > 50% are set to NULL
(probable splits, counted below) but the day stays in the volume baseline.

Tool choice: PySpark window functions, because the analysis needs per-ticker rolling statistics over 605,887 rows.

In [5]:
t = time.time()
w_t    = Window.partitionBy("ticker").orderBy("trade_date")
w_hist = w_t.rowsBetween(-WINDOW, -1)

base = (prices
    .withColumn("log_vol", F.log1p(F.col("volume").cast("double")))
    .withColumn("ret", F.col("close_price") / F.lag("close_price").over(w_t) - 1)
    .withColumn("abs_ret", F.when(F.abs("ret") <= MAX_ABS_RET, F.abs("ret")))
    .withColumn("next_abs_ret", F.lead("abs_ret").over(w_t))
    .withColumn("range_pct", (F.col("high_price") - F.col("low_price")) / F.col("close_price"))
    .withColumn("n_hist", F.count("log_vol").over(w_hist))
    .withColumn("mu", F.avg("log_vol").over(w_hist))
    .withColumn("sd", F.stddev("log_vol").over(w_hist))
    .withColumn("z", F.when((F.col("n_hist") == WINDOW) & (F.col("sd") > 0),
                            (F.col("log_vol") - F.col("mu")) / F.col("sd")))
    .withColumn("vol_class", F.when(F.col("z") >= Z_HIGH, "high")
                              .when(F.col("z") <= Z_LOW, "low")
                              .when(F.col("z").isNotNull(), "normal"))
    .cache())

n_base = base.count()
n_artefact = base.filter(F.abs("ret") > MAX_ABS_RET).count()
n_scored = base.filter(F.col("z").isNotNull()).count()
print("rows:", n_base, "| scored (>= %d prior days):" % WINDOW, n_scored,
      "| |return|>%.0f%% set to NULL:" % (MAX_ABS_RET * 100), n_artefact)

q1_summary = (base.filter(F.col("vol_class").isNotNull())
    .groupBy("vol_class")
    .agg(F.count("*").alias("n_days"),
         F.avg("abs_ret").alias("mean_abs_ret_same_day"),
         F.expr("percentile_approx(abs_ret, 0.5)").alias("median_abs_ret_same_day"),
         F.avg("next_abs_ret").alias("mean_abs_ret_next_day"),
         F.avg("range_pct").alias("mean_intraday_range"))
    .withColumn("share_of_scored_days", F.col("n_days") / F.lit(n_scored))
    .toPandas().set_index("vol_class").loc[["high", "normal", "low"]].reset_index())

q1_bucket = (base.filter(F.col("z").isNotNull())
    .withColumn("z_bucket", F.floor(F.greatest(F.least(F.col("z"), F.lit(4.0)), F.lit(-3.0)) * 2) / 2)
    .groupBy("z_bucket")
    .agg(F.count("*").alias("n_days"),
         F.avg("abs_ret").alias("mean_abs_ret_same_day"),
         F.avg("next_abs_ret").alias("mean_abs_ret_next_day"))
    .orderBy("z_bucket").toPandas())

q1_monthly = (base.filter(F.col("z").isNotNull())
    .withColumn("month", F.date_trunc("month", F.col("trade_date")).cast("date"))
    .groupBy("month")
    .agg(F.count("*").alias("n_scored_days"),
         F.sum((F.col("vol_class") == "high").cast("int")).alias("n_high"),
         F.sum((F.col("vol_class") == "low").cast("int")).alias("n_low"))
    .withColumn("share_high", F.col("n_high") / F.col("n_scored_days"))
    .orderBy("month").toPandas())
stage("q1_compute", t)
q1_summary

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

rows: 605887 | scored (>= 20 prior days): 595789 | |return|>50% set to NULL: 6
[q1_compute] 12.6 s
  vol_class  n_days  ...  mean_intraday_range  share_of_scored_days
0      high   32508  ...             0.033276              0.054563
1    normal  545647  ...             0.018370              0.915839
2       low   17634  ...             0.012803              0.029598

[3 rows x 7 columns]

In [6]:
normal_ret = float(q1_summary.loc[q1_summary.vol_class == "normal", "mean_abs_ret_same_day"].iloc[0])
q1_summary["same_day_abs_ret_vs_normal"] = q1_summary["mean_abs_ret_same_day"] / normal_ret

q1_summary.to_csv(f"{OUT_DIR}/q1_class_summary.csv", index=False)
q1_bucket.to_csv(f"{OUT_DIR}/q1_zbucket_summary.csv", index=False)
q1_monthly.to_csv(f"{OUT_DIR}/q1_monthly_flags.csv", index=False)
print(q1_summary.round(4).to_string(index=False))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

vol_class  n_days  mean_abs_ret_same_day  median_abs_ret_same_day  mean_abs_ret_next_day  mean_intraday_range  share_of_scored_days  same_day_abs_ret_vs_normal
     high   32508                 0.0255                   0.0179                 0.0148               0.0333                0.0546                      2.5760
   normal  545647                 0.0099                   0.0073                 0.0104               0.0184                0.9158                      1.0000
      low   17634                 0.0053                   0.0040                 0.0085               0.0128                0.0296                      0.5387

In [7]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
order = q1_summary["vol_class"].tolist()
bars = ax[0].bar(order, q1_summary["mean_abs_ret_same_day"] * 100, color=["#c0392b", "#7f8c8d", "#2980b9"])
for b, n in zip(bars, q1_summary["n_days"]):
    ax[0].text(b.get_x() + b.get_width() / 2, b.get_height(), f"n={n:,}", ha="center", va="bottom", fontsize=9)
ax[0].set_ylabel("Mean |daily return| (%)"); ax[0].set_xlabel("Volume class (vs own prior 20 days)")
ax[0].set_title("Q1: same-day price move by volume class")

ax[1].plot(q1_bucket["z_bucket"], q1_bucket["mean_abs_ret_same_day"] * 100, marker="o", label="same day")
ax[1].plot(q1_bucket["z_bucket"], q1_bucket["mean_abs_ret_next_day"] * 100, marker="s", label="next day")
ax[1].set_xlabel("Volume z-score bucket (clipped to [-3, 4])"); ax[1].set_ylabel("Mean |daily return| (%)")
ax[1].set_title("Q1: price move rises with unusual volume"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/q1_volume_class_and_zbucket.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(pd.to_datetime(q1_monthly["month"]), q1_monthly["share_high"] * 100)
ax.set_ylabel("% of scored days flagged high"); ax.set_xlabel("Month")
ax.set_title("Q1: share of ticker-days with unusually high volume, by month")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/q1_monthly_high_share.png", dpi=150); plt.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Q2. Does news volume / tone help explain subsequent market volatility?

Method: (1) market volatility per trading day = average |daily return| across tickers (PySpark); (2) join to the 150 GDELT dates on
`trade_date = available_date`; (3) split the joined days into terciles of `event_count` (news volume) and of `avg_tone` (sentiment);
(4) compare **next-trading-day** volatility across groups, with group sizes and spread. News on day t is compared with volatility on t+1,
because some day-t articles are published after the close. Same-day volatility is also reported and labelled contemporaneous.
(5) To ask whether news adds anything beyond price data alone, compare groups within low/high *today* volatility and test the
Spearman correlation of news with next-day volatility after removing the part explained by today's volatility.

Tool choice: PySpark for the daily aggregation, join and group statistics; the ~100-row result is analysed in pandas.
Observed relationships only; no causal claim is made.

In [8]:
t = time.time()
w_d = Window.orderBy("trade_date")
mkt = (base.filter(F.col("abs_ret").isNotNull())
       .groupBy("trade_date")
       .agg(F.avg("abs_ret").alias("mkt_vol"), F.countDistinct("ticker").alias("n_tickers"))
       .withColumn("mkt_vol_next", F.lead("mkt_vol").over(w_d)))

news = gdelt.withColumnRenamed("available_date", "trade_date")
joined = (mkt.join(news, "trade_date", "inner")
          .filter(F.col("mkt_vol_next").isNotNull())
          .withColumn("news_group", F.ntile(N_GROUPS).over(Window.orderBy("event_count")))
          .withColumn("tone_group", F.ntile(N_GROUPS).over(Window.orderBy("avg_tone")))
          .withColumn("today_vol_group", F.ntile(2).over(Window.orderBy("mkt_vol")))
          .withColumn("block", F.year("trade_date"))      # each GDELT block falls in a different year
          .cache())
n_joined = joined.count()
print("market-days:", mkt.count(), "| GDELT dates:", n_gdelt,
      "| joined trading days with a next day:", n_joined,
      "| GDELT dates dropped (weekends/holidays):", n_gdelt - n_joined)

def group_stats(col):
    return (joined.groupBy(col)
        .agg(F.count("*").alias("n_days"),
             F.min("event_count" if col == "news_group" else "avg_tone").alias("group_min"),
             F.max("event_count" if col == "news_group" else "avg_tone").alias("group_max"),
             F.avg("mkt_vol_next").alias("mean_next_day_vol"),
             F.expr("percentile_approx(mkt_vol_next, 0.5)").alias("median_next_day_vol"),
             F.stddev("mkt_vol_next").alias("sd_next_day_vol"),
             F.avg("mkt_vol").alias("mean_same_day_vol_contemporaneous"))
        .orderBy(col).toPandas())

q2_news = group_stats("news_group")
q2_tone = group_stats("tone_group")
q2_controlled = (joined.groupBy("today_vol_group", "news_group")
    .agg(F.count("*").alias("n_days"), F.avg("mkt_vol_next").alias("mean_next_day_vol"))
    .orderBy("today_vol_group", "news_group").toPandas())
q2_block_mix = (joined.groupBy("block")
    .agg(F.count("*").alias("n_days"),
         F.avg("mkt_vol").alias("mean_same_day_vol"),
         F.avg("avg_tone").alias("mean_tone"),
         F.avg("event_count").alias("mean_event_count"),
         F.sum((F.col("tone_group") == 1).cast("int")).alias("n_in_most_negative_tone_group"),
         F.sum((F.col("tone_group") == 3).cast("int")).alias("n_in_least_negative_tone_group"))
    .orderBy("block").toPandas())
daily = joined.select("trade_date", "block", "mkt_vol", "mkt_vol_next", "event_count", "avg_tone",
                      "news_group", "tone_group", "today_vol_group").orderBy("trade_date").toPandas()
stage("q2_compute", t)
q2_news

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

market-days: 1232 | GDELT dates: 150 | joined trading days with a next day: 100 | GDELT dates dropped (weekends/holidays): 50
[q2_compute] 5.0 s
   news_group  n_days  ...  sd_next_day_vol  mean_same_day_vol_contemporaneous
0           1      34  ...         0.007553                           0.013036
1           2      33  ...         0.006563                           0.015455
2           3      33  ...         0.005615                           0.012582

[3 rows x 8 columns]

In [9]:
import numpy as np
try:
    from scipy.stats import spearmanr
except ImportError:
    spearmanr = None

P_METHOD = "scipy spearmanr (asymptotic)" if spearmanr is not None else "permutation test, 10,000 shuffles, seed 42"

def spearman(x, y):
    if spearmanr is not None:
        r, p = spearmanr(x, y); return float(r), float(p)
    # no scipy: rank correlation + permutation p-value (numpy only)
    rx = pd.Series(np.asarray(x, dtype=float)).rank().to_numpy()
    ry = pd.Series(np.asarray(y, dtype=float)).rank().to_numpy()
    r = float(np.corrcoef(rx, ry)[0, 1])
    rng = np.random.default_rng(42)
    perm = np.array([np.corrcoef(rx, rng.permutation(ry))[0, 1] for _ in range(10000)])
    return r, float((np.sum(np.abs(perm) >= abs(r)) + 1) / (len(perm) + 1))
print("p-value method:", P_METHOD)

# part of next-day volatility NOT explained by today's volatility (simple linear fit)
X = np.column_stack([np.ones(len(daily)), daily["mkt_vol"].to_numpy()])
beta, *_ = np.linalg.lstsq(X, daily["mkt_vol_next"].to_numpy(), rcond=None)
daily["resid_next_vol"] = daily["mkt_vol_next"] - X @ beta
r2_today_only = 1 - daily["resid_next_vol"].var() / daily["mkt_vol_next"].var()

rows = []
for label, x, y in [
    ("event_count vs next-day vol",              daily["event_count"], daily["mkt_vol_next"]),
    ("avg_tone vs next-day vol",                 daily["avg_tone"],    daily["mkt_vol_next"]),
    ("event_count vs next-day vol | today vol",  daily["event_count"], daily["resid_next_vol"]),
    ("avg_tone vs next-day vol | today vol",     daily["avg_tone"],    daily["resid_next_vol"]),
    ("today vol vs next-day vol (volatility clustering)", daily["mkt_vol"], daily["mkt_vol_next"]),
    # within-block: remove each GDELT block's own average, so differences BETWEEN the three periods cannot drive the result
    ("WITHIN-BLOCK event_count vs next-day vol | today vol",
        daily["event_count"] - daily.groupby("block")["event_count"].transform("mean"),
        daily["resid_next_vol"] - daily.groupby("block")["resid_next_vol"].transform("mean")),
    ("WITHIN-BLOCK avg_tone vs next-day vol | today vol",
        daily["avg_tone"] - daily.groupby("block")["avg_tone"].transform("mean"),
        daily["resid_next_vol"] - daily.groupby("block")["resid_next_vol"].transform("mean")),
]:
    r, p = spearman(x, y)
    rows.append({"pair": label, "spearman_rho": round(r, 3), "p_value": round(p, 4), "n_days": len(daily),
                 "p_method": P_METHOD})  # p-values assume independent days; consecutive days are autocorrelated, so treat as optimistic
q2_corr = pd.DataFrame(rows)
print("R^2 of next-day vol explained by today's vol alone:", round(float(r2_today_only), 3))

q2_news.to_csv(f"{OUT_DIR}/q2_news_volume_groups.csv", index=False)
q2_tone.to_csv(f"{OUT_DIR}/q2_tone_groups.csv", index=False)
q2_controlled.to_csv(f"{OUT_DIR}/q2_controlled_by_today_vol.csv", index=False)
q2_corr.to_csv(f"{OUT_DIR}/q2_spearman_checks.csv", index=False)
daily.to_csv(f"{OUT_DIR}/q2_joined_daily.csv", index=False)
q2_block_mix.to_csv(f"{OUT_DIR}/q2_block_composition.csv", index=False)
print(q2_block_mix.round(4).to_string(index=False)); print()
print(q2_news.round(5).to_string(index=False)); print()
print(q2_tone.round(5).to_string(index=False)); print()
print(q2_controlled.round(5).to_string(index=False)); print()
print(q2_corr.to_string(index=False))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

p-value method: scipy spearmanr (asymptotic)
R^2 of next-day vol explained by today's vol alone: 0.192
 block  n_days  mean_same_day_vol  mean_tone  mean_event_count  n_in_most_negative_tone_group  n_in_least_negative_tone_group
  2015      62             0.0133    -2.0781       218681.8548                             16                              19
  2016      19             0.0188    -2.3617       222599.3158                             16                               0
  2017      19             0.0097    -1.8625       226461.4737                              2                              14

 news_group  n_days  group_min  group_max  mean_next_day_vol  median_next_day_vol  sd_next_day_vol  mean_same_day_vol_contemporaneous
          1      34     175886     213692            0.01333              0.01065          0.00755                            0.01304
          2      33     213969     227578            0.01488              0.01326          0.00656                          

In [10]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for a, col, title, labels in [(ax[0], "news_group", "News volume (GDELT event_count)", ["low", "mid", "high"]),
                              (ax[1], "tone_group", "News tone (GDELT avg_tone)", ["most negative", "mid", "least negative"])]:
    data = [daily.loc[daily[col] == g, "mkt_vol_next"] * 100 for g in range(1, N_GROUPS + 1)]
    a.boxplot(data)
    a.set_xticklabels([f"{l}\n(n={len(d)})" for l, d in zip(labels, data)])
    a.set_xlabel(title + " tercile"); a.set_title("Next-day market volatility by group")
ax[0].set_ylabel("Next-day mean |return| across tickers (%)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/q2_next_day_vol_by_news_group.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(6, 4.2))
sc = ax.scatter(daily["mkt_vol"] * 100, daily["mkt_vol_next"] * 100, c=daily["event_count"], cmap="viridis", s=22)
plt.colorbar(sc, label="GDELT event_count")
ax.set_xlabel("Today's market volatility (%)"); ax.set_ylabel("Next-day market volatility (%)")
ax.set_title("Q2: volatility clustering dominates; colour = news volume")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/q2_today_vs_next_day_vol.png", dpi=150); plt.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Q3. Where is the Task 3 model most and least reliable?

Inputs: the **2017 test predictions** saved by the two final Task 3 models (`ticker, trade_date, target, prediction, risk_score`) joined to the
Task 2 model-ready test features on `(ticker, trade_date)`. Nothing is retrained, no cutoff is changed and nothing here is tuned on test results.

- **Models.** *volatility*: will next-week volatility be in the top quartile (1) or not (0); *direction*: will the close rise over the next five sessions.
- **Baselines (same rows).** *rule* = the Task 3 historical rule (volatility: 20-day volatility above its **training** 75th percentile; direction: past 5-day return above 0);
  *majority* = always predict the training majority class (volatility: 0 by construction of the label; direction: from training labels).
- **Subgroups.** calendar month of 2017, and quartile of 20-day volatility (cut points from the **training** split only).
- **Metrics.** balanced accuracy (primary, because the volatility classes are imbalanced) with a 95% normal-approximation interval, and AUC. AUC is computed in Spark from average ranks.
  Accuracy alone is not used to judge the model. Inside a volatility quartile the volatility rule predicts a single class, so its balanced accuracy is 0.5 by construction;
  volatility-quartile comparisons therefore use AUC, and the code flags any group where a prediction is constant.
- Model scores are classification diagnostics, **not economic returns** and not evidence that the model was investable.

Tool choice: PySpark window functions and grouped aggregates, because every subgroup metric is computed over 100k+ prediction rows; only the small result tables go to pandas for charts.


In [11]:
# ---- Q3 inputs (change here only; override with environment variables if the paths differ) ----
T3_BASE = os.environ.get("G12_T3_BASE", BUCKET + "/task3")
MODEL_READY = os.environ.get("G12_MODEL_READY", T3_BASE + "/task2/model_ready_v1")   # Task 2 splits: training / validation / testing
Q3_MODELS = {
    "volatility": {"pred": os.environ.get("G12_PRED_VOL", T3_BASE + "/task3final/volatility/20261118T151322Z/predictions"),
                   "rule_col": "volatility_20d", "expected_rows": 123476,
                   "rule_text": "high volatility if 20-day volatility > training 75th percentile",
                   "positive_text": "high next-week volatility"},
    "direction":  {"pred": os.environ.get("G12_PRED_DIR", T3_BASE + "/task3final/direction/20261118T150036Z/predictions"),
                   "rule_col": "return_5d", "expected_rows": 126001,
                   "rule_text": "rise if past 5-day return > 0",
                   "positive_text": "price rises over next week"},
}

t = time.time()
train_feat = spark.read.parquet(MODEL_READY + "/training").select("ticker", "trade_date", "label", "volatility_20d", "return_5d")
test_feat  = spark.read.parquet(MODEL_READY + "/testing").select("ticker", "trade_date", "volatility_20d", "return_5d")

# cut points come from TRAINING data only
VOL_Q = train_feat.approxQuantile("volatility_20d", [0.25, 0.5, 0.75], 0.001)
Q3_RULE_CUT = {"volatility": VOL_Q[2], "direction": 0.0}
Q3_MAJORITY = {"volatility": 0.0,   # the volatility label is 1 for the top training quartile, so class 0 is the training majority
               "direction": float(train_feat.groupBy("label").count().orderBy(F.desc("count")).first()["label"])}
print("volatility_20d training quartiles:", [round(v, 5) for v in VOL_Q], "| direction majority class:", Q3_MAJORITY["direction"])

def load_scored(name):
    cfg = Q3_MODELS[name]
    p = spark.read.parquet(cfg["pred"]).select(
        "ticker", F.to_date("trade_date").alias("trade_date"),
        F.col("target").cast("double"), F.col("prediction").cast("double"), F.col("risk_score").cast("double"))
    n = p.count()
    n_dup = p.groupBy("ticker", "trade_date").count().filter("count > 1").count()
    j = p.join(test_feat.withColumn("trade_date", F.to_date("trade_date")), ["ticker", "trade_date"], "left")
    n_unmatched = j.filter(F.col("volatility_20d").isNull() | F.col("return_5d").isNull()).count()
    ok = (n == cfg["expected_rows"]) and n_dup == 0 and n_unmatched == 0
    print(f"{name}: prediction rows {n:,} (expected {cfg['expected_rows']:,}) | duplicate keys {n_dup} | unmatched to features {n_unmatched} ->", "PASS" if ok else "FAIL")
    assert ok, f"{name}: prediction file does not match the expected test set"
    rc = cfg["rule_col"]
    j = (j.withColumn("base_score", F.col(rc))
          .withColumn("base_pred", (F.col(rc) > F.lit(float(Q3_RULE_CUT[name]))).cast("double"))
          .withColumn("maj_pred", F.lit(Q3_MAJORITY[name])).withColumn("maj_score", F.lit(Q3_MAJORITY[name]))
          .withColumn("month", F.date_format("trade_date", "yyyy-MM"))
          .withColumn("vol_bucket", F.when(F.col("volatility_20d") <= VOL_Q[0], "V1 lowest")
                                     .when(F.col("volatility_20d") <= VOL_Q[1], "V2")
                                     .when(F.col("volatility_20d") <= VOL_Q[2], "V3")
                                     .otherwise("V4 highest"))
          .withColumn("all", F.lit("all 2017"))
          .cache())
    j.count()
    return j, n

scored, Q3_ROWS = {}, {}
for name in Q3_MODELS:
    scored[name], Q3_ROWS[name] = load_scored(name)
stage("q3_load", t)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

volatility_20d training quartiles: [0.00944, 0.01215, 0.01606] | direction majority class: 1.0
volatility: prediction rows 123,476 (expected 123,476) | duplicate keys 0 | unmatched to features 0 -> PASS
direction: prediction rows 126,001 (expected 126,001) | duplicate keys 0 | unmatched to features 0 -> PASS
[q3_load] 10.1 s

In [12]:
import numpy as np

def grouped_metrics(df, group_col, pred_col, score_col):
    """Confusion counts + AUC per group, all computed in Spark. AUC = Mann-Whitney U from average ranks (ties handled)."""
    g = F.col(group_col)
    c = df.groupBy(group_col).agg(
        F.count("*").alias("n"),
        F.sum((F.col("target") == 1).cast("int")).alias("pos"),
        F.sum(((F.col("target") == 1) & (F.col(pred_col) == 1)).cast("int")).alias("tp"),
        F.sum(((F.col("target") == 0) & (F.col(pred_col) == 1)).cast("int")).alias("fp"),
        F.sum(((F.col("target") == 1) & (F.col(pred_col) == 0)).cast("int")).alias("fn"),
        F.sum(((F.col("target") == 0) & (F.col(pred_col) == 0)).cast("int")).alias("tn"))
    w, wt = Window.partitionBy(group_col).orderBy(score_col), Window.partitionBy(group_col, score_col)
    r = (df.withColumn("rk", F.rank().over(w)).withColumn("ties", F.count("*").over(wt))
           .withColumn("avg_rk", F.col("rk") + (F.col("ties") - 1) / 2.0)
           .groupBy(group_col).agg(F.sum(F.when(F.col("target") == 1, F.col("avg_rk"))).alias("rank_sum_pos")))
    o = c.join(r, group_col).orderBy(group_col).toPandas().rename(columns={group_col: "group"})
    P, N = o["pos"].astype(float), (o["n"] - o["pos"]).astype(float)
    ok = (P > 0) & (N > 0)
    o["auc"] = np.where(ok, (o["rank_sum_pos"] - P * (P + 1) / 2) / (P * N).where(ok, 1.0), np.nan)
    o["recall"] = np.where(P > 0, o["tp"] / P.where(P > 0, 1.0), np.nan)
    o["specificity"] = np.where(N > 0, o["tn"] / N.where(N > 0, 1.0), np.nan)
    o["balanced_accuracy"] = (o["recall"] + o["specificity"]) / 2
    o["accuracy"] = (o["tp"] + o["tn"]) / o["n"]
    o["ba_ci95"] = 1.96 * 0.5 * np.sqrt(o["recall"] * (1 - o["recall"]) / P.where(P > 0, np.nan)
                                        + o["specificity"] * (1 - o["specificity"]) / N.where(N > 0, np.nan))
    o["positive_rate"] = o["pos"] / o["n"]
    # a constant prediction inside a group (e.g. a threshold rule on the same variable used to form the group) gives balanced accuracy 0.5 by construction
    o["single_class_prediction"] = ((o["tp"] + o["fp"]) == 0) | ((o["fn"] + o["tn"]) == 0)
    return o.drop(columns=["rank_sum_pos"])

VARIANTS = {"model": ("prediction", "risk_score"), "rule baseline": ("base_pred", "base_score"), "majority baseline": ("maj_pred", "maj_score")}
t = time.time()
overall_rows, month_rows, bucket_rows = [], [], []
for name, df in scored.items():
    for variant, (pc, sc_) in VARIANTS.items():
        o = grouped_metrics(df, "all", pc, sc_);  o.insert(0, "variant", variant); o.insert(0, "model", name); overall_rows.append(o)
        if variant == "majority baseline":
            continue                                    # majority is constant, so it adds nothing by month or volatility group
        m = grouped_metrics(df, "month", pc, sc_);      m.insert(0, "variant", variant); m.insert(0, "model", name); month_rows.append(m)
        b = grouped_metrics(df, "vol_bucket", pc, sc_); b.insert(0, "variant", variant); b.insert(0, "model", name); bucket_rows.append(b)
q3_overall = pd.concat(overall_rows, ignore_index=True)
q3_month   = pd.concat(month_rows, ignore_index=True)
q3_bucket  = pd.concat(bucket_rows, ignore_index=True)
stage("q3_compute", t)

for df_, fn in [(q3_overall, "q3_overall_metrics.csv"), (q3_month, "q3_month_metrics.csv"), (q3_bucket, "q3_volbucket_metrics.csv")]:
    df_.to_csv(f"{OUT_DIR}/{fn}", index=False)

show = ["model", "variant", "group", "n", "positive_rate", "balanced_accuracy", "ba_ci95", "auc", "accuracy"]
print("OVERALL 2017 TEST (compare with the Task 3 metrics file):"); print(q3_overall[show + ["tn", "fp", "fn", "tp"]].round(4).to_string(index=False)); print()
print("BY VOLATILITY QUARTILE (cut points from training):"); print(q3_bucket[show].round(4).to_string(index=False)); print()
print("BY MONTH:"); print(q3_month[show].round(4).to_string(index=False))

# gap between the model and the rule baseline per subgroup (positive = model better)
def gaps(tbl, key):
    a = tbl[tbl.variant == "model"].set_index(["model", "group"])
    b = tbl[tbl.variant == "rule baseline"].set_index(["model", "group"]).reindex(a.index)
    ba_ok = ~(a["single_class_prediction"] | b["single_class_prediction"])      # balanced accuracy is only comparable if neither side predicts one class
    g = pd.DataFrame({"model": a.index.get_level_values(0), "group": a.index.get_level_values(1), "subgroup_type": key,
                      "n": a["n"].values, "auc_gain_vs_rule": (a["auc"] - b["auc"]).values,
                      "ba_gain_vs_rule": np.where(ba_ok.values, (a["balanced_accuracy"] - b["balanced_accuracy"]).values, np.nan)})
    return g
q3_gaps = pd.concat([gaps(q3_month, "month"), gaps(q3_bucket, "vol_bucket")], ignore_index=True)
q3_gaps.to_csv(f"{OUT_DIR}/q3_model_minus_rule_gaps.csv", index=False)
for name in Q3_MODELS:
    for metric in ["auc_gain_vs_rule", "ba_gain_vs_rule"]:
        gg = q3_gaps[(q3_gaps.model == name)].dropna(subset=[metric])
        best, worst = gg.loc[gg[metric].idxmax()], gg.loc[gg[metric].idxmin()]
        print(f"{name} | {metric}: model better in {(gg[metric] > 0).sum()} of {len(gg)} comparable subgroups | "
              f"largest gain {best[metric]:+.3f} ({best.subgroup_type} {best.group}, n={int(best.n):,}) | "
              f"largest loss {worst[metric]:+.3f} ({worst.subgroup_type} {worst.group}, n={int(worst.n):,})")
rb = q3_bucket[q3_bucket.variant == "rule baseline"].groupby("model")["single_class_prediction"].agg(["sum", "count"])
for name, r in rb.iterrows():
    if r["sum"]:
        print(f"NOTE: {name}: the rule baseline predicts a single class in {int(r['sum'])} of {int(r['count'])} volatility-quartile groups, "
              "so its balanced accuracy there is 0.5 by construction; volatility-quartile comparisons use AUC.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[q3_compute] 15.5 s
OVERALL 2017 TEST (compare with the Task 3 metrics file):
     model           variant    group      n  positive_rate  balanced_accuracy  ba_ci95    auc  accuracy     tn    fp    fn    tp
volatility             model all 2017 123476         0.1813             0.6401   0.0033 0.7455    0.8075  91263  9831 13936  8446
volatility     rule baseline all 2017 123476         0.1813             0.6197   0.0034 0.7026    0.7738  87079 14015 13919  8463
volatility majority baseline all 2017 123476         0.1813             0.5000   0.0000 0.5000    0.8187 101094     0 22382     0
 direction             model all 2017 126001         0.5634             0.5105   0.0027 0.5132    0.4920  36134 18876 45134 25857
 direction     rule baseline all 2017 126001         0.5634             0.4941   0.0028 0.4932    0.5019  23771 31239 31520 39471
 direction majority baseline all 2017 126001         0.5634             0.5000   0.0000 0.5000    0.5634      0 55010     0 70991

BY VOLATILI

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for a, name in zip(axes, Q3_MODELS):
    for variant, mk, col in [("model", "o", "#c0392b"), ("rule baseline", "s", "#7f8c8d")]:
        d = q3_month[(q3_month.model == name) & (q3_month.variant == variant)]
        a.errorbar(d["group"], d["balanced_accuracy"], yerr=d["ba_ci95"], marker=mk, color=col, capsize=2, label=variant)
    a.axhline(0.5, color="k", lw=0.8, ls="--"); a.set_title(f"Q3 {name}: balanced accuracy by month, 2017 test")
    a.set_xlabel("Month (95% interval)"); a.tick_params(axis="x", rotation=60)
axes[0].set_ylabel("Balanced accuracy (0.5 = chance)"); axes[0].legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/q3_balanced_accuracy_by_month.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True)
for a, name in zip(axes, Q3_MODELS):
    d = q3_bucket[q3_bucket.model == name]; groups = sorted(d["group"].unique()); x = np.arange(len(groups)); wd = 0.38
    for k, (variant, col) in enumerate([("model", "#c0392b"), ("rule baseline", "#7f8c8d")]):
        dv = d[d.variant == variant].set_index("group").loc[groups]
        a.bar(x + (k - 0.5) * wd, dv["auc"], wd, color=col, label=variant)
    n_lab = d[d.variant == "model"].set_index("group").loc[groups, "n"]
    a.set_xticks(x); a.set_xticklabels([f"{g}\n(n={int(n):,})" for g, n in zip(groups, n_lab)])
    a.axhline(0.5, color="k", lw=0.8, ls="--"); a.set_ylim(0.4, None)
    a.set_title(f"Q3 {name}: AUC by 20-day volatility quartile"); a.set_xlabel("Volatility quartile (cut points from training data)")
axes[0].set_ylabel("AUC (0.5 = chance)"); axes[0].legend()
fig.text(0.5, -0.02, "AUC is used here because a volatility threshold rule predicts one class inside each volatility quartile (balanced accuracy fixed at 0.5).",
         ha="center", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/q3_auc_by_volatility_quartile.png", dpi=150, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for a, name in zip(axes, Q3_MODELS):
    r = q3_overall[(q3_overall.model == name) & (q3_overall.variant == "model")].iloc[0]
    cm = np.array([[r.tn, r.fp], [r.fn, r.tp]], dtype=float); pct = cm / cm.sum(axis=1, keepdims=True)
    a.imshow(pct, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            a.text(j, i, f"{int(cm[i, j]):,}\n{pct[i, j]:.0%} of row", ha="center", va="center", color="white" if pct[i, j] > 0.5 else "black")
    a.set_xticks([0, 1]); a.set_xticklabels(["predicted 0", "predicted 1"]); a.set_yticks([0, 1]); a.set_yticklabels(["actual 0", "actual 1"])
    a.set_title(f"Q3 {name}: confusion matrix, 2017 test")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/q3_confusion_matrices.png", dpi=150); plt.show()

Q3_MANIFEST = {"prediction_paths": {k: v["pred"] for k, v in Q3_MODELS.items()}, "model_ready_path": MODEL_READY, "prediction_rows": Q3_ROWS,
               "volatility_20d_training_quartiles": VOL_Q, "rule_cutoffs": Q3_RULE_CUT, "majority_class": Q3_MAJORITY}


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Run manifest (evidence for Appendix B)

In [14]:
manifest = {
    "run_id": RUN_ID, "spark_version": spark.version, "on_cluster": ON_CLUSTER,
    "inputs": {"prices": PRICES_PATH, "gdelt_daily_features": GDELT_PATH, "task3_predictions": {k: v["pred"] for k, v in Q3_MODELS.items()}},
    "row_counts": {"prices": n_prices, "gdelt_dates": n_gdelt, "scored_ticker_days": n_scored,
                   "returns_nulled_as_artefacts": n_artefact, "q2_joined_days": n_joined},
    "q3": Q3_MANIFEST,
    "parameters": {"window": WINDOW, "z_high": Z_HIGH, "z_low": Z_LOW, "max_abs_ret": MAX_ABS_RET, "groups": N_GROUPS},
    "stage_seconds": TIMINGS, "total_seconds": round(time.time() - T0, 1),
    "outputs": sorted(os.listdir(OUT_DIR)),
}
with open(f"{OUT_DIR}/run_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)
print(json.dumps(manifest, indent=2, default=str))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{
  "run_id": "20261120T075457Z",
  "spark_version": "3.5.5-amzn-1",
  "on_cluster": true,
  "inputs": {
    "prices": "s3://inf2006-g12-2500837/task2/cleaned/prices",
    "gdelt_daily_features": "s3://inf2006-g12-2500837/task2/cleaned/gdelt/daily_features",
    "task3_predictions": {
      "volatility": "s3://inf2006-g12-2500837/task3/task3final/volatility/20261118T151322Z/predictions",
      "direction": "s3://inf2006-g12-2500837/task3/task3final/direction/20261118T150036Z/predictions"
    }
  },
  "row_counts": {
    "prices": 605887,
    "gdelt_dates": 150,
    "scored_ticker_days": 595789,
    "returns_nulled_as_artefacts": 6,
    "q2_joined_days": 100
  },
  "q3": {
    "prediction_paths": {
      "volatility": "s3://inf2006-g12-2500837/task3/task3final/volatility/20261118T151322Z/predictions",
      "direction": "s3://inf2006-g12-2500837/task3/task3final/direction/20261118T150036Z/predictions"
    },
    "model_ready_path": "s3://inf2006-g12-2500837/task3/task2/model_ready_v1",


In [15]:
import os
print(OUT_DIR, RUN_ID)
print(sorted(os.listdir(OUT_DIR)))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

evidence/task4 20261120T075457Z
['q1_class_summary.csv', 'q1_monthly_flags.csv', 'q1_monthly_high_share.png', 'q1_volume_class_and_zbucket.png', 'q1_zbucket_summary.csv', 'q2_block_composition.csv', 'q2_controlled_by_today_vol.csv', 'q2_joined_daily.csv', 'q2_news_volume_groups.csv', 'q2_next_day_vol_by_news_group.png', 'q2_spearman_checks.csv', 'q2_today_vs_next_day_vol.png', 'q2_tone_groups.csv', 'q3_auc_by_volatility_quartile.png', 'q3_balanced_accuracy_by_month.png', 'q3_confusion_matrices.png', 'q3_model_minus_rule_gaps.csv', 'q3_month_metrics.csv', 'q3_overall_metrics.csv', 'q3_volbucket_metrics.csv', 'run_manifest.json']

In [16]:
import subprocess
S3_EVIDENCE = f"{BUCKET}/evidence/task4_{RUN_ID}"
r = subprocess.run(["aws", "s3", "cp", "--recursive", OUT_DIR, S3_EVIDENCE], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

.9 KiB (3.4 MiB/s) with 6 file(s) remaining
upload: evidence/task4/q3_overall_metrics.csv to s3://inf2006-g12-2500837/evidence/task4_20261120T075457Z/q3_overall_metrics.csv
Completed 474.4 KiB/610.9 KiB (3.4 MiB/s) with 5 file(s) remaining
Completed 595.2 KiB/610.9 KiB (4.2 MiB/s) with 5 file(s) remaining
upload: evidence/task4/q3_balanced_accuracy_by_month.png to s3://inf2006-g12-2500837/evidence/task4_20261120T075457Z/q3_balanced_accuracy_by_month.png
Completed 595.2 KiB/610.9 KiB (4.2 MiB/s) with 4 file(s) remaining
Completed 597.6 KiB/610.9 KiB (4.1 MiB/s) with 4 file(s) remaining
upload: evidence/task4/run_manifest.json to s3://inf2006-g12-2500837/evidence/task4_20261120T075457Z/run_manifest.json
Completed 597.6 KiB/610.9 KiB (4.1 MiB/s) with 3 file(s) remaining
Completed 607.0 KiB/610.9 KiB (4.0 MiB/s) with 3 file(s) remaining
upload: evidence/task4/q3_month_metrics.csv to s3://inf2006-g12-2500837/evidence/task4_20261120T075457Z/q3_month_metrics.csv
Completed 607.0 KiB/610.9 KiB 